### Evaluation

Import

In [1]:
from matplotlib import pyplot as plt
import pandas as pd



Compare Load Forecast

In [64]:
#Compare Load Forecast to Actual Load with NMAE hourly
def calculate_nmae_load_forecast_to_actual_load_hourly(
    load_forecast: pd.DataFrame,
    actual_load: pd.DataFrame,
    load_forecast_col: str = 'Load',
    actual_load_col: str = 'Load'
) -> float:
    """
    Calculates the Normalized Mean Absolute Error (NMAE) between forecast and actual hourly load.

    Parameters:
        load_forecast (pd.DataFrame): DataFrame with datetime index and forecast values in 'load_forecast_col'.
        actual_load (pd.DataFrame): DataFrame with datetime index and actual values in 'actual_load_col'.
        load_forecast_col (str): Name of the forecast column. Default is 'Load'.
        actual_load_col (str): Name of the actual load column. Default is 'Load'.

    Returns:
        float: NMAE in percent (%), based on sum-normalized absolute error.
    """

    # Extract Series
    forecast_series = load_forecast[load_forecast_col]
    actual_series = actual_load[actual_load_col]

    # Ensure both series have the same index
    if not forecast_series.index.equals(actual_series.index):
        raise ValueError("Forecast and actual series must have the same datetime index.")
    # Zeitzonen vereinheitlichen → wir nehmen naive Zeit (ohne +00:00)
    actual_series.index = actual_series.index.tz_convert(None) if actual_series.index.tz else actual_series.index
    forecast_series.index = forecast_series.index.tz_convert(None) if forecast_series.index.tz else forecast_series.index

    
    # Calculate NMAE
    absolute_errors = (forecast_series - actual_series).abs()
    nmae = absolute_errors.sum() / actual_series.abs().sum() * 100  # in %

    print(abs(absolute_errors).sum(), actual_series.abs().sum())
    return nmae

Test

In [66]:
#Actual_Load from D:\Dokumente\Studium\20242025 WS\WattWise\Repo\wattWise\data\test_data_actual_load_profile_2020.csv
actual_load = pd.read_csv(r'D:\Dokumente\Studium\20242025 WS\WattWise\Repo\wattWise\data\test_data_actual_load_profile_2020.csv', sep=';', index_col='date')
#only SFH4_P_TOT
actual_load = actual_load[['SFH4_P_TOT']]
actual_load.index = pd.to_datetime(actual_load.index)
actual_load.index = actual_load.index.tz_convert(None)

load_forecast = pd.read_csv(r'D:\Dokumente\Studium\20242025 WS\WattWise\Repo\wattWise\data\test_data_load_forecast_profile_2020.csv', sep=';', index_col='Time')
# Load Forecast drop index containing 31.12.2020 (letzte 24h)
load_forecast = load_forecast.drop(load_forecast.index[-24:])
load_forecast.index = pd.to_datetime(load_forecast.index)

print(type(actual_load), type(load_forecast))
print(actual_load.shape, load_forecast.shape)

<class 'pandas.core.frame.DataFrame'> <class 'pandas.core.frame.DataFrame'>
(8760, 1) (8760, 1)


In [44]:
actual_load

,SFH4_P_TOT
date,
2020-01-01 00:00:00+00:00,0.338549
2020-01-01 01:00:00+00:00,0.393269
2020-01-01 02:00:00+00:00,0.412591
2020-01-01 03:00:00+00:00,0.342366
2020-01-01 04:00:00+00:00,0.251848
...,...
2020-12-30 19:00:00+00:00,1.168154
2020-12-30 20:00:00+00:00,0.402383
2020-12-30 21:00:00+00:00,0.392819


In [56]:
load_forecast

,Load
Time,
2020-01-01 00:00:00,0.235293
2020-01-01 01:00:00,0.237022
2020-01-01 02:00:00,0.282202
2020-01-01 03:00:00,0.323197
2020-01-01 04:00:00,0.221215
...,...
2020-12-30 19:00:00,0.416956
2020-12-30 20:00:00,0.473081
2020-12-30 21:00:00,0.463537


In [67]:
#NMAE
nmae = calculate_nmae_load_forecast_to_actual_load_hourly(load_forecast, actual_load, load_forecast_col="Load", actual_load_col="SFH4_P_TOT")

print(f"NMAE: {nmae:.2f}%")

1405.8486729479505 2999.999999999999
NMAE: 46.86%
